In [ ]:
import sys
sys.path.append('../')

import numpy as np 
import pandas as pd 

from sklearn.metrics import cohen_kappa_score, accuracy_score,balanced_accuracy_score

from plotly import express as px

from tutoriales.utils import plot_confusion_matrix, get_artifact_filename

import os

from json import loads

from joblib import load, dump

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

c:\Users\u581537.TELECOM.000\.conda\envs\ldi2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths
BASE_DIR = '../'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "work/cleaned/train_clean.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

In [9]:
study_lgb = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name="lgbm_modelo_2__variables_completas",
                            load_if_exists = True)


lgb_dataset = load(os.path.join(PATH_TO_TEMP_FILES,get_artifact_filename(study_lgb,'test')))

[I 2026-04-30 17:51:19,136] Using an existing study with name 'lgbm_modelo_2__variables_completas' instead of creating a new one.


ValueError: Record does not exist.

In [ ]:
#lgb_dataset

In [3]:
MODEL_NAME = '01 DistilBert'
MODEL_VERSION = '3.0'

study_bert = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)

bert_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_bert,'test')))

[I 2026-04-30 17:48:32,777] Using an existing study with name '01 DistilBert_3.0' instead of creating a new one.


In [7]:
#bert_dataset

In [4]:
# Cargar el modelo ResNet
MODEL_NAME_RESNET = '04 ResNet Augment'
MODEL_VERSION_RESNET = '1.0.0'

study_resnet = optuna.create_study(
    direction='maximize',
    storage="sqlite:///../work/optuna_artifacts/db.sqlite3",
    study_name=f'{MODEL_NAME_RESNET}_{MODEL_VERSION_RESNET}',
    load_if_exists=True
)

[I 2026-04-30 17:48:36,080] Using an existing study with name '04 ResNet Augment_1.0.0' instead of creating a new one.


In [5]:
import sys, numpy.core
# Shim: permite cargar joblib guardados con numpy >= 2.0 en entornos con numpy 1.x
sys.modules.setdefault("numpy._core", numpy.core)
for _sub in ["numeric", "multiarray", "umath", "fromnumeric", "arrayprint", "strings"]:
    mod = getattr(numpy.core, _sub, numpy.core)
    sys.modules.setdefault(f"numpy._core.{_sub}", mod)


In [6]:
resnet_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_04 ResNet Augment_1.0.0_1.joblib'))

In [7]:
merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(bert_dataset[['PetID', 'pred']].rename({'pred':'bert_pred_score'},axis=1),
                  on='PetID', how='outer')

NameError: name 'lgb_dataset' is not defined

In [ ]:
# Unir ResNet al dataframe fusionado
merged_datasets = merged_datasets.merge(
    resnet_dataset[['PetID', 'pred']].rename({'pred': 'resnet_pred_score'}, axis=1),
    on='PetID', how='outer'
)

In [ ]:
# Limpiar nulos (rellenar con arrays de ceros si algún modelo no tiene predicción para un PetID)
merged_datasets['resnet_pred_score'] = [np.zeros(5) if type(i) is float else i for i in merged_datasets['resnet_pred_score']]

In [ ]:
# Optimización con Optuna para 3 pesos
def objective(trial):
    # Definir pesos para los tres modelos
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_bert = trial.suggest_float('w_bert', 0.0, 1.0)
    w_resnet = trial.suggest_float('w_resnet', 0.0, 1.0)
    
    # Normalización
    total_w = w_lgb + w_bert + w_resnet
    
    # Cálculo vectorizado para mayor velocidad
    # Convertimos las columnas de scores en una matriz 3D o sumamos directamente
    lgb_scores = np.stack(merged_datasets['lgb_pred_score'].values)
    bert_scores = np.stack(merged_datasets['bert_pred_score'].values)
    resnet_scores = np.stack(merged_datasets['resnet_pred_score'].values)
    
    combined_scores = (
        (w_lgb / total_w) * lgb_scores + 
        (w_bert / total_w) * bert_scores + 
        (w_resnet / total_w) * resnet_scores
    )
    
    preds_final = np.argmax(combined_scores, axis=1)
    
    return cohen_kappa_score(merged_datasets['AdoptionSpeed'], preds_final, weights='quadratic')

In [ ]:
# Ejecutar el estudio
STORAGE_URL = "sqlite:///../work/db.sqlite3"
study_blend = optuna.create_study(
    direction='maximize',
    storage=STORAGE_URL,
    study_name="Ensemble_LGB_BERT_ResNet",
    load_if_exists=True
)
study_blend.optimize(objective, n_trials=100)

In [ ]:
# Resultados
best_params = study_blend.best_params
sum_best_w = sum(best_params.values())

print(f"Mejor Kappa: {study_blend.best_value:.4f}")
print(f"Pesos óptimos: {best_params}")

In [ ]:
# Crear la columna de predicción final optimizada
merged_datasets['blend_pred_score'] = [
    (best_params['w_lgb'] / sum_best_w) * r['lgb_pred_score'] +
    (best_params['w_bert'] / sum_best_w) * r['bert_pred_score'] +
    (best_params['w_resnet'] / sum_best_w) * r['resnet_pred_score']
    for _, r in merged_datasets.iterrows()
]

In [ ]:
from optuna.visualization import plot_param_importances

# Generar el gráfico de importancia de hiperparámetros (pesos)
fig = plot_param_importances(study_blend)

fig.update_layout(
    title="Importancia de los Modelos en el Ensamble",
    xaxis_title="Importancia relativa",
    yaxis_title="Modelo (Pesos)",
    template="plotly_white"
)

fig.show()


In [ ]:
from optuna.visualization import plot_contour

In [ ]:
fig_contour = plot_contour(study_blend, params=['w_lgb', 'w_bert', 'w_resnet'])

fig_contour.update_layout(
    title="Interacción de Pesos y Rendimiento (Kappa)",
    width=900,
    height=800
)

fig_contour.show()

In [ ]:
STORAGE_URL = "sqlite:///../work/db.sqlite3"

study_blend = optuna.create_study(
    direction='maximize', 
    storage=STORAGE_URL, 
    study_name="Ensemble_LGB_BERT_ResNet",
    load_if_exists=True # Esto permite pausar y continuar la optimización luego
)

study_blend.optimize(objective, n_trials=100)

In [ ]:
# Guardar el resultado final en un archivo temporal y subirlo a Optuna
final_filename = "merged_predictions_optimized.joblib"
dump(merged_datasets, final_filename)


In [ ]:
# Subir el archivo como artefacto del mejor trial
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)
upload_artifact(
    trial=study_blend.best_trial, 
    file_path=final_filename, 
    artifact_store=artifact_store
)


In [ ]:
#-------------------------------------------------------------

In [ ]:
#merged_datasets

In [ ]:
#merged_datasets['blend_pred_score'] = [r['lgb_pred_score']+r['bert_pred_score'] for i,r in merged_datasets.iterrows()]

In [ ]:
#merged_datasets['lgb_pred_score']

In [ ]:
#merged_datasets['lgb_pred'] = [r.argmax() for r in merged_datasets['lgb_pred_score']]
#merged_datasets['bert_pred'] = [r.argmax() for r in merged_datasets['bert_pred_score']]
#merged_datasets['blended_pred'] = [r.argmax() for r in merged_datasets['blend_pred_score']]

In [ ]:
#plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
#                      merged_datasets['lgb_pred'], 
#                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
#                                                                    merged_datasets['lgb_pred'], 
#                                                                    weights='quadratic')))

In [ ]:
#plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
#                      merged_datasets['bert_pred'], 
#                    title = 'Bert Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
#                                                                   merged_datasets['bert_pred'], 
#                                                                    weights='quadratic')))



In [ ]:
#plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
#                     merged_datasets['blended_pred'], 
#                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
#                                                                    merged_datasets['blended_pred'], 
#                                                                    weights='quadratic')))
